# Dark Manifold Virtual Cell — Full Prototype Run

This notebook runs every prototype end-to-end and captures each one's output
into a JSON file for later analysis.

**Two tracks:**
- **Cell-simulator substrate** (P0–P10): atom conservation, Syn3A biochemistry,
  learned rates, spatial simulator, LSODA integration.
- **Architecture investigation** (P11–P15): neural PDEs, unitary evolution,
  gauge/memory/rule mechanisms tested against matched-parameter baselines.

**Plus:** P8b and P8c (scaled-up permutation-invariant networks for full Syn3A).

Total run time on GPU Colab: ~30–45 minutes. All outputs stream to
`dmvc_outputs.json` so you can download and inspect after the run.

Designed for Colab with GPU. Falls back to CPU if no GPU available.

## 1. Setup — install, clone, copy

In [ ]:
# Install dependencies
!pip install -q numpy python-libsbml scipy torch

In [ ]:
# Clone the Syn3A data repo (needed for P2 onward)
import os
if not os.path.exists('/content/Minimal_Cell'):
    !git clone --depth 1 https://github.com/Luthey-Schulten-Lab/Minimal_Cell.git /content/Minimal_Cell
print('data ok:', os.path.exists('/content/Minimal_Cell/CME_ODE/model_data/iMB155_NoH2O.xml'))

### Upload the prototype bundle

Upload `dmvc_prototypes.tar.gz` (from the latest output), then extract.

In [ ]:
# Upload dmvc_prototypes.tar.gz via the file uploader
from google.colab import files
print("Click 'Choose Files' and upload dmvc_prototypes.tar.gz")
uploaded = files.upload()

In [ ]:
# Extract to /content/dmvc
!mkdir -p /content/dmvc
!tar -xzf dmvc_prototypes.tar.gz -C /content/
!ls /content/dmvc | head -40

In [ ]:
# Patch SBML_PATH: the prototypes look for the SBML relative to their
# hard-coded /home/claude/dmvc path. Redirect to the Colab clone path.
import re, os, glob
SBML = '/content/Minimal_Cell/CME_ODE/model_data/iMB155_NoH2O.xml'
assert os.path.exists(SBML), f"SBML not found at {SBML}"
for f in glob.glob('/content/dmvc/prototype_*.py'):
    with open(f) as fh: src = fh.read()
    new = re.sub(
        r"SBML_PATH\s*=\s*['\"][^'\"]*['\"]",
        f'SBML_PATH = "{SBML}"', src)
    if new != src:
        with open(f, 'w') as fh: fh.write(new)
        print('patched', os.path.basename(f))

In [ ]:
# Put /content/dmvc on the Python path so cross-imports work
import sys
if '/content/dmvc' not in sys.path:
    sys.path.insert(0, '/content/dmvc')

# Also cd there so relative paths in prototypes work
import os
os.chdir('/content/dmvc')
print('cwd:', os.getcwd())

## 2. GPU check

P8b and P8c auto-detect CUDA and will use GPU if available. Other
neural-network prototypes run on CPU via subprocess and are small enough
(< 2 min each) that this is fine.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}')
print(f'device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('(CPU fallback -- P8c will be slower but should still run)')

In [ ]:
# Prototypes run as subprocesses, so they don't inherit this notebook's
# device state. P8b and P8c have built-in GPU detection; other neural
# prototypes (P7, P7b, P8, P10, P11-P15) run on CPU but are small and fast.
print('GPU will be used by: P8b, P8c (built-in detection)')
print('CPU will be used by: P7, P7b, P8, P10, P11, P12, P13, P14, P15')
print('No GPU needed for: P0-P6, P9 (pure numpy/scipy)')

## 3. Output capture infrastructure

Every prototype runs through a helper that:
- captures stdout and stderr
- times the run
- writes each result into `/content/dmvc_outputs.json` incrementally
  (so a mid-run crash still preserves earlier results)

In [ ]:
import json, time, subprocess, os, sys

OUTPUTS_PATH = '/content/dmvc_outputs.json'

def load_outputs():
    if os.path.exists(OUTPUTS_PATH):
        with open(OUTPUTS_PATH) as f:
            return json.load(f)
    return {}

def save_outputs(data):
    with open(OUTPUTS_PATH, 'w') as f:
        json.dump(data, f, indent=2)

def run_prototype(name, script_path, timeout=1800):
    outputs = load_outputs()
    print(f'=== {name} ===')
    print(f'running {script_path} ...')
    t0 = time.time()
    try:
        result = subprocess.run(
            [sys.executable, script_path],
            cwd='/content/dmvc',
            capture_output=True, text=True, timeout=timeout,
            env={**os.environ, 'PYTHONPATH': '/content/dmvc'})
        elapsed = time.time() - t0
        entry = {
            'name': name,
            'script': script_path,
            'returncode': result.returncode,
            'elapsed_s': round(elapsed, 2),
            'stdout': result.stdout,
            'stderr': result.stderr,
            'timed_out': False,
        }
    except subprocess.TimeoutExpired as e:
        elapsed = time.time() - t0
        entry = {
            'name': name,
            'script': script_path,
            'returncode': -1,
            'elapsed_s': round(elapsed, 2),
            'stdout': (e.stdout.decode() if e.stdout else ''),
            'stderr': (e.stderr.decode() if e.stderr else '') + f'\n*** TIMED OUT after {timeout}s ***',
            'timed_out': True,
        }
    outputs[name] = entry
    save_outputs(outputs)
    tail = entry['stdout'].strip().split('\n')[-25:]
    for line in tail:
        print(line)
    print(f'[{name}] rc={entry["returncode"]} elapsed={entry["elapsed_s"]}s')
    print()
    return entry

print(f'outputs will stream to {OUTPUTS_PATH}')
if os.path.exists(OUTPUTS_PATH):
    os.remove(OUTPUTS_PATH)
    print('cleared previous outputs')

## 4. Cell-simulator track: P0 → P10

### P0 — atom conservation via subspace projection

Smallest test. 3 atom types, random reactions, projection keeps atoms conserved.

In [ ]:
run_prototype('P0_atom_conservation', '/content/dmvc/prototype_p0.py')

### P1 — stamp+flavor embeddings, balanced reactions

In [ ]:
run_prototype('P1_stamps', '/content/dmvc/prototype_p1.py')

### P2 — load real Syn3A SBML

In [ ]:
run_prototype('P2_syn3a', '/content/dmvc/prototype_p2_syn3a.py')

### P2b — rebalance with H⁺ and H₂O

In [ ]:
run_prototype('P2b_rebalance', '/content/dmvc/prototype_p2b_rebalance.py')

### P3b — compartment-aware stamps

In [ ]:
run_prototype('P3b_compartment_stamps', '/content/dmvc/prototype_p3b_stamps.py')

### P4 — well-mixed Syn3A kinetics

In [ ]:
run_prototype('P4_kinetics', '/content/dmvc/prototype_p4_kinetics.py')

### P4b — kinetics coupled to Ψ field

In [ ]:
run_prototype('P4b_kinetics_coupled', '/content/dmvc/prototype_p4b_kinetics_coupled.py')

### P5 — boundary fluxes (open system)

In [ ]:
run_prototype('P5_boundary', '/content/dmvc/prototype_p5_boundary.py')

### P6 — physiological tuning

In [ ]:
run_prototype('P6_physiological', '/content/dmvc/prototype_p6_physiological.py', timeout=300)

### P7 — first learned rate predictor

In [ ]:
run_prototype('P7_learned_rates', '/content/dmvc/prototype_p7_learned_rates.py', timeout=300)

### P7b — tuned learned rate predictor

In [ ]:
run_prototype('P7b_tuned', '/content/dmvc/prototype_p7b_tuned.py', timeout=600)

### P8 — permutation-invariant network (original)

Scenario 1 (glycolysis held-out): should pass at ~47% zero-shot error.
Scenario 2 (full Syn3A): the original null — collapses to zero because of
rate-scale disparity. Kept here for reference.

In [ ]:
run_prototype('P8_perm_invariant', '/content/dmvc/prototype_p8_perm_invariant.py', timeout=900)

### P8b — full Syn3A scale, fixed loss design

Per-reaction log-scale normalization + species embeddings fix the P8 Scenario 2
collapse. Network learns direction + rough magnitude across all 221 reactions
(median ~60% error) — substantial progress over the null.

In [ ]:
run_prototype('P8b_full_syn3a', '/content/dmvc/prototype_p8b_full_syn3a.py', timeout=900)

### P8c — scaled-up P8b

Hidden 512, embedding 16, more samples, longer training. This is the run
that most benefits from GPU. Expected: median error drops further; if it
passes T1/T2 the single-network Syn3A claim has clean support.

In [ ]:
run_prototype('P8c_scaled', '/content/dmvc/prototype_p8c_scaled.py', timeout=1800)

### P9 — LSODA stiff integration

In [ ]:
run_prototype('P9_lsoda', '/content/dmvc/prototype_p9_lsoda.py', timeout=600)

### P10 — learned rates in spatial simulator (hybrid)

In [ ]:
run_prototype('P10_learned_spatial', '/content/dmvc/prototype_p10_learned_spatial.py', timeout=900)

## 5. Architecture track: P11 → P15

### P11 — neural PDE (Fisher-KPP)

Local conv + periodic padding learns a nonlinear reaction-diffusion PDE.
All 4 tests pass; rollout error < 1% on held-out initial conditions.

In [ ]:
run_prototype('P11_neural_pde', '/content/dmvc/prototype_p11_neural_pde.py', timeout=600)

### P12 — gauge/connection field (NULL)

Gauge factorization on complex Ginzburg-Landau. Baseline absorbs gauge
structure implicitly; learned A collapses to near-constant. Filed as
honest null.

In [ ]:
run_prototype('P12_gauge', '/content/dmvc/prototype_p12_gauge.py', timeout=600)

### P13 — structural unitary evolution (strong pass)

Schrödinger equation target. Structurally-unitary model beats baseline
by 3× on rollout accuracy. Renormalization alone doesn't close the gap —
unitarity does more than preserve norm.

In [ ]:
run_prototype('P13_unitary', '/content/dmvc/prototype_p13_unitary.py', timeout=600)

### P14 — memory bank (NULL)

Retrieval-augmented attention over past states. Task wasn't memory-hungry
enough; attention collapsed to uniform. Filed as honest null.

In [ ]:
run_prototype('P14_memory', '/content/dmvc/prototype_p14_memory.py', timeout=600)

### P15 — rule discovery (NULL)

Mixture-of-K rules on multi-regime reaction-diffusion. Mode collapse to
one effective rule; rule-region agreement at chance. Third null in the
modular-factorization pattern.

In [ ]:
run_prototype('P15_rule_discovery', '/content/dmvc/prototype_p15_rule_discovery.py', timeout=600)

## 6. Run summary

In [ ]:
# Summarize what's in dmvc_outputs.json
import json
with open('/content/dmvc_outputs.json') as f:
    outputs = json.load(f)

total_time = sum(v['elapsed_s'] for v in outputs.values())
print(f'Captured {len(outputs)} prototype runs. Total wall clock: {total_time:.1f}s')
print()
print(f'{"prototype":<28s} {"rc":>4s} {"time (s)":>10s} {"status":>10s}')
print('-' * 60)
for name, entry in outputs.items():
    rc = entry['returncode']
    status = 'OK' if rc == 0 else ('TIMEOUT' if entry.get('timed_out') else 'FAIL')
    print(f'{name:<28s} {rc:>4d} {entry["elapsed_s"]:>10.1f} {status:>10s}')

In [ ]:
# Show the last 20 lines of stdout for each failed run
for name, entry in outputs.items():
    if entry['returncode'] != 0:
        print(f'=== FAILURE: {name} ===')
        tail = entry['stdout'].strip().split('\n')[-20:]
        for line in tail:
            print(line)
        if entry['stderr'].strip():
            print('--- stderr ---')
            err_tail = entry['stderr'].strip().split('\n')[-10:]
            for line in err_tail:
                print(line)
        print()

## 7. Download outputs

The JSON contains every prototype's full stdout and stderr. Download it
to your machine so the run is preserved even if this Colab session dies.

In [ ]:
from google.colab import files
files.download('/content/dmvc_outputs.json')

## 8. Notes

- **P8c is the headline run on GPU.** It's the one that was memory-starved
  on the original container. On a Colab T4 or better it should finish in
  a few minutes and either pass T1/T2 (strong result) or give clean
  diagnostics (see the `p8c_results.npz` written to /content/dmvc/).
- **P12, P14, P15 are filed nulls.** They'll run and show non-passing
  results; that's the expected outcome, captured for reproducibility.
- The JSON is ~1–2 MB once full. Fits comfortably in memory.